## Prompt Engineering with LangChain - Complete Practice Notebook (VS Code)
Welcome to the consolidated Prompt Engineering practice notebook. This file combines all exercises and coding tasks from the entire module.

### <a id="table-of-contents"></a>📋 Table of Contents
* [Chapter 1: Introduction to LangChain](#chapter-1-introduction-to-langchain)
* [Chapter 2: Prompt Templates & Message Structures](#chapter-2-prompt-templates-and-message-structures)
* [Chapter 3: LCEL & Runnables](#chapter-3-lcel-and-runnables)
* [Chapter 4: Output Parsers](#chapter-4-output-parsers)
* [Chapter 5: Model Hyperparameters](#chapter-5-model-hyperparameters)
* [Chapter 6: Few-Shot & Sequential Prompting](#chapter-6-few-shot-and-sequential-prompting)

### 🛠[] Setup Environment & Load API Keys

In [ ]:
# Load local environment variables from .env
import os
from dotenv import load_dotenv
load_dotenv()

### <a id="chapter-1-introduction-to-langchain"></a>📖 Chapter 1: Introduction to LangChain

#### 1.1 OpenAI Direct API (Conceptual)

In [ ]:
# from openai import OpenAI
# client = OpenAI()
# response = client.chat.completions.create(
#     model='gpt-4o',
#     messages=[{'role': 'user', 'content': 'Explain LangChain.'}]
# )

#### 1.2 Google Gemini Direct API (Conceptual)

In [ ]:
# import google.generativeai as genai
# model = genai.GenerativeModel('gemini-2.5-flash')
# response = model.generate_content('Explain LangChain.')

#### 1.3 Unified LangChain Invocation (Active)

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")
response = llm.invoke("Explain LangChain in 1 sentence.")
print(response.content)

#### 1.4 Initializing with Groq

In [ ]:
groq_llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
response = groq_llm.invoke("Explain why developers use virtual environments in Python.")
print(response.content)

#### Exercises for Chapter 1
##### Exercise 1: Dual-Provider Setup & Comparison
Write a script that prompts **both** Groq (`llama-3.3-70b-versatile`) and Google (`gemini-2.5-flash`) with the question `"State the main goal of prompt engineering in 5 words."` and prints the response from each model.

<details>
<summary><b>Click to view solution</b></summary>

```python
groq_llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
gemini_llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai")
prompt = "State the main goal of prompt engineering in 5 words."
print("--- Groq Response ---")
print(groq_llm.invoke(prompt).content.strip())
print("\n--- Gemini Response ---")
print(gemini_llm.invoke(prompt).content.strip())
```
</details>

In [ ]:
# Write your code here

[Back to Top](#table-of-contents)

### <a id="chapter-2-prompt-templates-and-message-structures"></a>📖 Chapter 2: Prompt Templates & Message Structures

#### 2.1 String PromptTemplate Example

In [ ]:
from langchain_core.prompts import PromptTemplate
string_template = PromptTemplate.from_template("Explain {topic} in 1 sentence.")
prompt_val = string_template.invoke({"topic": "object-oriented programming"})
print(prompt_val.to_string())

#### 2.2 ChatPromptTemplate Example

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful translation assistant."),
    ("human", "Translate the following phrase into {language}: '{phrase}'")
])
chat_prompt = chat_template.invoke({"language": "Spanish", "phrase": "Hello, how are you?"})
print(chat_prompt)

#### Exercises for Chapter 2
##### Exercise 1: Recipe Generator
Define a `PromptTemplate` that takes an `ingredients` list (e.g., "tomato, cheese, basil") and a `cuisine` type (e.g., "Italian"), and prompts the model to generate a recipe.

<details>
<summary><b>Click to view solution</b></summary>

```python
recipe_template = PromptTemplate.from_template(
    "Create a traditional {cuisine} recipe using these ingredients: {ingredients}."
)
prompt = recipe_template.invoke({"cuisine": "Italian", "ingredients": "tomato, cheese, basil"})
print(prompt.to_string())
```
</details>

In [ ]:
# Write your code here

[Back to Top](#table-of-contents)

### <a id="chapter-3-lcel-and-runnables"></a>📖 Chapter 3: LCEL & Runnables

#### 3.1 Basic QA Chain with LCEL

In [ ]:
joke_prompt = PromptTemplate.from_template("Tell me a short joke about {topic}.")
joke_chain = joke_prompt | llm
response = joke_chain.invoke({"topic": "programming"})
print(response.content)

#### 3.2 Streaming Response Example

In [ ]:
for chunk in joke_chain.stream({"topic": "Python generators"}):
    print(chunk.content, end="", flush=True)

#### Exercises for Chapter 3
##### Exercise 1: Marketing Slogan Generator
Compose a chain using LCEL linking a PromptTemplate (`"Create a catchy marketing slogan for {product_name} targeting {target_audience}. Output ONLY the slogan."`) and your Chat Model. Invoke it for `product_name="EcoWater Bottle"` and `target_audience="fitness enthusiasts"`.

<details>
<summary><b>Click to view solution</b></summary>

```python
slogan_prompt = PromptTemplate.from_template(
    "Create a catchy marketing slogan for {product_name} targeting {target_audience}. Output ONLY the slogan."
)
slogan_chain = slogan_prompt | llm
result = slogan_chain.invoke({
    "product_name": "EcoWater Bottle",
    "target_audience": "fitness enthusiasts"
})
print(result.content)
```
</details>

In [ ]:
# Write your code here

[Back to Top](#table-of-contents)

### <a id="chapter-4-output-parsers"></a>📖 Chapter 4: Output Parsers

#### 4.1 StrOutputParser and CommaSeparatedListOutputParser

In [ ]:
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser

str_parser = StrOutputParser()
list_parser = CommaSeparatedListOutputParser()

print("String parsing:", str_parser.invoke(response))
print("List parsing:", list_parser.invoke("Apple, Banana, Orange"))

#### Exercises for Chapter 4
##### Exercise 1: Comma Separated List Parsing
Create a prompt template that requests the model to list the top 3 programming languages for web development, and chain it with the `CommaSeparatedListOutputParser` to obtain a Python list.

<details>
<summary><b>Click to view solution</b></summary>

```python
parser = CommaSeparatedListOutputParser()
prompt = PromptTemplate(
    template="List 3 top programming languages for {use_case} as a comma-separated list.",
    input_variables=["use_case"]
)
chain = prompt | llm | parser
result = chain.invoke({"use_case": "web development"})
print(result)
```
</details>

In [ ]:
# Write your code here

[Back to Top](#table-of-contents)

### <a id="chapter-5-model-hyperparameters"></a>📖 Chapter 5: Model Hyperparameters

#### 5.1 Restricting Max Tokens and Adjusting Temperature

In [ ]:
llm_strict = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0.0, max_tokens=10)
response = llm_strict.invoke("Complete: A stitch in time saves...")
print(response.content)

#### Exercises for Chapter 5
##### Exercise 1: Configuring Parameters in LangChain
Configure a chat model using `init_chat_model` with a temperature of `0.0` and a max token limit of `100` to answer the question: `"State the value of Pi to 10 decimal places."`

<details>
<summary><b>Click to view solution</b></summary>

```python
llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0.0, max_tokens=100)
response = llm.invoke("State the value of Pi to 10 decimal places.")
print(response.content)
```
</details>

In [ ]:
# Write your code here

[Back to Top](#table-of-contents)

### <a id="chapter-6-few-shot-and-sequential-prompting"></a>📖 Chapter 6: Few-Shot & Sequential Prompting

#### 6.1 Few-Shot Math Assistant Example

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate

examples = [
    {"question": "2 + 2", "answer": "4"},
    {"question": "3 + 3", "answer": "6"}
]
example_prompt = PromptTemplate(input_variables=["question", "answer"], template="Question: {question}\nAnswer: {answer}")
few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question: {input}\nAnswer:",
    input_variables=["input"]
)
print(few_shot.format(input="10 + 10"))

#### Exercises for Chapter 6
##### Exercise 1: Few-Shot Entity Extraction
Build a Few-Shot prompt classifying person and company names. Examples: `"John works at Google." -> {"person": "John", "company": "Google"}`, `"Alice joined Microsoft." -> {"person": "Alice", "company": "Microsoft"}`. Run the query on: `"Bob works at Amazon."`

<details>
<summary><b>Click to view solution</b></summary>

```python
examples = [
    {"text": "John works at Google.", "output": '{"person": "John", "company": "Google"}'},
    {"text": "Alice joined Microsoft.", "output": '{"person": "Alice", "company": "Microsoft"}'}
]
example_prompt = PromptTemplate(input_variables=["text", "output"], template="Text: {text}\nOutput: {output}")
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Text: {input}\nOutput:",
    input_variables=["input"]
)
formatted = few_shot_prompt.format(input="Bob works at Amazon.")
print(llm.invoke(formatted).content)
```
</details>

In [ ]:
# Write your code here